In [1]:
import torch 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import os

In [2]:
os.getcwd()

'/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_TREE_SEGMENTATION'

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
from Unet_model import UNet

In [8]:
my_model = UNet()
my_model.load_state_dict(torch.load("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_TREE_SEGMENTATION/results/unet/unet_tree_50_2.pth"))
my_model.to(device=device)

UNet(
  (encoder): ModuleList(
    (0): Sequential(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (1): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (2): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (3): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (4): Sequential(
      (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

In [9]:
root_dir = '/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_TREE_SEGMENTATION/TREE_DATASET/test'

val_images_dir = os.path.join(root_dir, 'images')
val_masks_dir = os.path.join(root_dir, 'masks')

In [10]:
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import transforms
import cv2

In [11]:
# Custom Dataset
class RooftopDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_filenames = sorted(os.listdir(image_dir))
        self.transform = transform
    
    def __len__(self):
        return len(self.image_filenames)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_filenames[idx])
        mask_path = os.path.join(self.mask_dir, self.image_filenames[idx])
        
        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = mask.astype(np.float32) / 255.0
        
        if self.transform:
            image = self.transform(image)
            mask = transforms.ToTensor()(mask).unsqueeze(0)  # Ensure mask shape [1, H, W]
        
        return image, mask.squeeze(0)  # Ensure mask shape [H, W]

# Transformations
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((1024, 1024)),
    transforms.ToTensor()
])

# Load datasets
val_dataset = RooftopDataset(val_images_dir, val_masks_dir, transform)

val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)


In [12]:
os.getcwd()

'/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_TREE_SEGMENTATION'

In [13]:
os.chdir("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection")
os.getcwd()

'/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection'

In [14]:
from Solar_Rooftop_Detection.accuracy import compute_metrics

In [15]:
# Evaluation

def evaluate(model, val_loader):
    model.eval()
    result_data = []
    # iou_scores = []
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            # print(outputs.shape)
            preds = (outputs > 0.5).float()
            
            # one  = preds[0].squeeze(0).cpu().numpy()
            # plt.imshow(one, cmap="gray")
            # plt.tight_layout()
            # plt.show()
            
            # original = masks[0].squeeze(0).cpu().numpy()
            # plt.imshow(original, cmap="gray")
            # plt.tight_layout()
            # plt.show()

            metrics = compute_metrics(preds, masks)
            result_data.append(metrics)

    return result_data

In [16]:
accuracy_result = evaluate(my_model, val_loader)
accuracy_result

[[np.float64(0.24192992108063166),
  0.3896031764338569,
  0.7996854186058044,
  0.5220341721926508,
  0.3107669921225894,
  0.24192992108063166,
  0.3896031764338569,
  0.5220341721926508,
  0.3107669921225894,
  0.0],
 [np.float64(0.34579165777384185),
  0.5138858689997194,
  0.8384907245635986,
  0.7349667524326299,
  0.3950525204926737,
  0.34579165777384185,
  0.5138858689997194,
  0.7349667524326299,
  0.3950525204926737,
  0.0],
 [np.float64(0.27135710787917644),
  0.4268778712093611,
  0.8032866716384888,
  0.6538511667743935,
  0.31687879491799714,
  0.27135710787917644,
  0.4268778712093611,
  0.6538511667743935,
  0.31687879491799714,
  0.0],
 [np.float64(0.18347528901692814),
  0.3100618842144726,
  0.8807645440101624,
  0.48628514988159416,
  0.2275872823636868,
  0.18347528901692814,
  0.3100618842144726,
  0.48628514988159416,
  0.2275872823636868,
  0.0],
 [np.float64(0.16809347507526196),
  0.28780825963338497,
  0.897369384765625,
  0.6107404228128659,
  0.18826314579

In [17]:
csv_path = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_TREE_SEGMENTATION/results/unet/unet_validation_results_1.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

In [18]:
import pandas as pd 

metrics_df = pd.DataFrame(accuracy_result, columns=["pixel_iou", "pixel_dice", "pixel_accuracy", "pixel_precision", "pixel_recall", "region_iou", "region_dice", "region_precision", "region_recall", "region_success_accuracy"])
metrics_df.to_csv(csv_path, index=False)

In [20]:
metrics_df.mean()

pixel_iou                  0.249266
pixel_dice                 0.396487
pixel_accuracy             0.866894
pixel_precision            0.579099
pixel_recall               0.306533
region_iou                 0.249266
region_dice                0.396487
region_precision           0.579099
region_recall              0.306533
region_success_accuracy    0.000000
dtype: float64